# Football Match Prediction — Live Testing

This notebook is for **prospective testing**: generate predictions before upcoming matches, save them, and evaluate them after the matches finish.

The model uses:
- Historical team attacking and defensive strengths
- Recent form (last 5 matches)
- Independent Poisson goal distributions
- Fixed parameters selected during historical validation: `form_weight=0.2`, `n_form=5`

**Important:** Keep the model parameters fixed during this experiment. Generate and save predictions before kickoff. Do not edit old predictions after seeing results.

## 1. Imports and settings

Change `DATA_PATH` to the location of your historical results CSV. The CSV must contain `Date`, `HomeTeam`, `AwayTeam`, `FTHG`, `FTAG`, and `FTR`.

`FTR` should use `H` for home win, `D` for draw, and `A` for away win.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import poisson
from sklearn.metrics import log_loss, accuracy_score, classification_report, confusion_matrix

DATA_PATH = "your_historical_matches.csv"

PREDICTIONS_PATH = "live_predictions.csv"

FORM_WEIGHT = 0.2
N_FORM = 5
MAX_GOALS = 10

## 2. Load and prepare historical results

Only completed matches should be in this dataset. Update this file as new results become available.

The model will filter historical data by each fixture's date, so results dated on or after the fixture date are not used.

In [ ]:
df = pd.read_csv(DATA_PATH)

required_columns = [
    "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"
]

missing = set(required_columns) - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["FTHG"] = pd.to_numeric(df["FTHG"], errors="coerce")
df["FTAG"] = pd.to_numeric(df["FTAG"], errors="coerce")
df["FTR"] = df["FTR"].astype("string").str.strip().str.upper()

df = df.dropna(subset=required_columns).copy()
df["FTHG"] = df["FTHG"].astype(int)
df["FTAG"] = df["FTAG"].astype(int)
df = df[df["FTR"].isin(["H", "D", "A"])].copy()
df = df.sort_values("Date").reset_index(drop=True)

print("Historical matches:", len(df))
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Teams:", df["HomeTeam"].nunique(), "home-team names;",
      df["AwayTeam"].nunique(), "away-team names")

FileNotFoundError: [Errno 2] No such file or directory: 'your_historical_matches.csv'

## 3. Model functions

These are the same model components used in the historical project. Team strengths and form are recalculated using only matches before the fixture date.

In [ ]:
def strengths_before(date, matches):
    past = matches[matches["Date"] < date]

    if len(past) == 0:
        raise ValueError("No historical matches available before this fixture date.")

    league_home_avg = past["FTHG"].mean()
    league_away_avg = past["FTAG"].mean()
    avg_team = (league_home_avg + league_away_avg) / 2

    if avg_team <= 0:
        raise ValueError("Historical average goals must be positive.")

    teams = sorted(set(past["HomeTeam"]) | set(past["AwayTeam"]))
    rows = []

    for team in teams:
        home = past[past["HomeTeam"] == team]
        away = past[past["AwayTeam"] == team]

        scored = home["FTHG"].sum() + away["FTAG"].sum()
        conceded = home["FTAG"].sum() + away["FTHG"].sum()
        played = len(home) + len(away)

        rows.append({
            "Team": team,
            "Attack": (scored / played) / avg_team,
            "Defence": (conceded / played) / avg_team
        })

    return pd.DataFrame(rows).set_index("Team")


def recent_form(team, date, matches, n=5):
    past = matches[matches["Date"] < date]

    games = past[
        (past["HomeTeam"] == team) |
        (past["AwayTeam"] == team)
    ].tail(n)

    if len(games) == 0:
        return {
            "games": 0,
            "goals_scored": np.nan,
            "goals_conceded": np.nan,
            "points": np.nan
        }

    is_home = (games["HomeTeam"] == team).to_numpy()

    scored = np.where(is_home, games["FTHG"], games["FTAG"])
    conceded = np.where(is_home, games["FTAG"], games["FTHG"])

    points = np.where(scored > conceded, 3,
                      np.where(scored == conceded, 1, 0))

    return {
        "games": len(games),
        "goals_scored": scored.mean(),
        "goals_conceded": conceded.mean(),
        "points": points.mean()
    }


def predict_match(
    home_team,
    away_team,
    date,
    matches,
    form_weight=FORM_WEIGHT,
    n_form=N_FORM,
    max_goals=MAX_GOALS
):
    date = pd.Timestamp(date)
    past = matches[matches["Date"] < date]

    if len(past) == 0:
        raise ValueError("No historical matches available before fixture date.")

    league_home_avg = past["FTHG"].mean()
    league_away_avg = past["FTAG"].mean()
    avg_team = (league_home_avg + league_away_avg) / 2

    if min(league_home_avg, league_away_avg, avg_team) <= 0:
        raise ValueError("Invalid historical goal averages.")

    strengths = strengths_before(date, matches)

    def blended(team):
        if team not in strengths.index:
            attack, defence = 1.0, 1.0
        else:
            attack = strengths.loc[team, "Attack"]
            defence = strengths.loc[team, "Defence"]

        form = recent_form(team, date, matches, n_form)

        if form["games"] == n_form:
            recent_attack = form["goals_scored"] / avg_team
            recent_defence = form["goals_conceded"] / avg_team

            attack = (1 - form_weight) * attack + form_weight * recent_attack
            defence = (1 - form_weight) * defence + form_weight * recent_defence

        return attack, defence

    attack_h, defence_h = blended(home_team)
    attack_a, defence_a = blended(away_team)

    expected_home = max(league_home_avg * attack_h * defence_a, 1e-10)
    expected_away = max(league_away_avg * attack_a * defence_h, 1e-10)

    goals = np.arange(max_goals + 1)
    home_probs = poisson.pmf(goals, expected_home)
    away_probs = poisson.pmf(goals, expected_away)

    score_matrix = np.outer(home_probs, away_probs)
    score_matrix /= score_matrix.sum()

    p_home = np.tril(score_matrix, -1).sum()
    p_draw = np.trace(score_matrix)
    p_away = np.triu(score_matrix, 1).sum()

    return expected_home, expected_away, p_home, p_draw, p_away

## 4. Enter upcoming fixtures

Add fixtures with their scheduled kickoff dates. Use the exact team names that appear in your historical dataset.

The date should be the fixture's local date/time if your source includes times. This model currently uses dates rather than kickoff timestamps to filter historical matches, so it will exclude all matches played on the same calendar date. That is a conservative choice that avoids accidentally using same-day results.

In [ ]:
upcoming_fixtures = pd.DataFrame([
    # Replace these examples with real upcoming fixtures:
    # {"Date": "2026-11-25", "HomeTeam": "Arsenal", "AwayTeam": "Chelsea"},
])

upcoming_fixtures["Date"] = pd.to_datetime(upcoming_fixtures["Date"])

if upcoming_fixtures.empty:
    print("Add upcoming fixtures to the upcoming_fixtures DataFrame, then rerun this section.")
else:
    print(upcoming_fixtures)

## 5. Generate and save predictions

This also records a historical-frequency baseline calculated using information available before each fixture. Saving those baseline probabilities now is important: do not recalculate them later using a dataset that contains future results.

The code refuses to create a second prediction for a fixture already present in the CSV, helping prevent accidental overwriting.

In [ ]:
def historical_baseline(date, matches):
    past = matches[matches["Date"] < date]

    if past.empty:
        raise ValueError("No historical outcomes available for baseline.")

    probs = (
        past["FTR"]
        .value_counts(normalize=True)
        .reindex(["H", "D", "A"], fill_value=0)
        .to_numpy(dtype=float)
    )

    # Small smoothing prevents zero-probability outcomes.
    probs = probs + 1e-6
    return probs / probs.sum()


def make_live_predictions(fixtures, matches, output_path=PREDICTIONS_PATH):
    if fixtures.empty:
        print("No fixtures supplied.")
        return pd.DataFrame()

    existing = (
        pd.read_csv(output_path)
        if os.path.exists(output_path)
        else pd.DataFrame()
    )

    new_rows = []

    for _, fixture in fixtures.iterrows():
        date = pd.Timestamp(fixture["Date"])
        home = fixture["HomeTeam"]
        away = fixture["AwayTeam"]

        key_exists = False
        if not existing.empty:
            existing["Date"] = pd.to_datetime(existing["Date"])
            key_exists = (
                (existing["Date"] == date) &
                (existing["HomeTeam"] == home) &
                (existing["AwayTeam"] == away)
            ).any()

        if key_exists:
            print(f"Already recorded; skipping: {home} vs {away} ({date.date()})")
            continue

        eh, ea, pH, pD, pA = predict_match(
            home, away, date, matches
        )

        bH, bD, bA = historical_baseline(date, matches)

        new_rows.append({
            "Date": date,
            "HomeTeam": home,
            "AwayTeam": away,
            "PredictionTimestampUTC": pd.Timestamp.now(tz="UTC").isoformat(),
            "ExpectedHomeGoals": eh,
            "ExpectedAwayGoals": ea,
            "pH": pH,
            "pD": pD,
            "pA": pA,
            "Baseline_pH": bH,
            "Baseline_pD": bD,
            "Baseline_pA": bA,
            "Actual": pd.NA
        })

    new_predictions = pd.DataFrame(new_rows)

    if new_predictions.empty:
        print("No new predictions generated.")
        return new_predictions

    combined = pd.concat([existing, new_predictions], ignore_index=True)
    combined.to_csv(output_path, index=False)

    print(f"Saved {len(new_predictions)} new predictions to {output_path}")
    return new_predictions


live_predictions = make_live_predictions(upcoming_fixtures, df)

live_predictions

## 6. After matches finish: enter actual results

After a fixture has finished, enter its final home and away goals below. The function updates the saved prediction file but leaves the original predicted probabilities unchanged.

Use `H`, `D`, or `A` for the actual outcome.

In [ ]:
def add_actual_results(results, output_path=PREDICTIONS_PATH):
    if not os.path.exists(output_path):
        raise FileNotFoundError("No saved live predictions file found.")

    saved = pd.read_csv(output_path)
    saved["Date"] = pd.to_datetime(saved["Date"])

    for result in results:
        date = pd.Timestamp(result["Date"])
        home = result["HomeTeam"]
        away = result["AwayTeam"]
        home_goals = int(result["FTHG"])
        away_goals = int(result["FTAG"])

        actual = "H" if home_goals > away_goals else (
            "D" if home_goals == away_goals else "A"
        )

        mask = (
            (saved["Date"] == date) &
            (saved["HomeTeam"] == home) &
            (saved["AwayTeam"] == away)
        )

        if mask.sum() != 1:
            print(f"Could not uniquely find: {home} vs {away} ({date.date()})")
            continue

        saved.loc[mask, "FTHG"] = home_goals
        saved.loc[mask, "FTAG"] = away_goals
        saved.loc[mask, "Actual"] = actual

    saved.to_csv(output_path, index=False)
    print("Results updated.")


# Example: replace with completed fixtures after the final whistle.
completed_results = [
    # {
    #     "Date": "2026-11-25",
    #     "HomeTeam": "Arsenal",
    #     "AwayTeam": "Chelsea",
    #     "FTHG": 2,
    #     "FTAG": 1
    # }
]

# Run after entering completed results:
# add_actual_results(completed_results)

## 7. Evaluate prospective predictions

This evaluates only fixtures for which actual results have been entered. It compares the Poisson model with the baseline probabilities saved **at prediction time**.

A small live sample will be noisy. Accumulate a meaningful number of matches before drawing conclusions.

In [ ]:
def evaluate_live_predictions(output_path=PREDICTIONS_PATH):
    if not os.path.exists(output_path):
        raise FileNotFoundError("No live predictions file found.")

    data = pd.read_csv(output_path)
    completed = data.dropna(subset=["Actual"]).copy()

    if completed.empty:
        print("No completed predictions to evaluate yet.")
        return None

    y = completed["Actual"].astype(str).to_numpy()

    model_probs = completed[["pH", "pD", "pA"]].to_numpy(dtype=float)
    baseline_probs = completed[
        ["Baseline_pH", "Baseline_pD", "Baseline_pA"]
    ].to_numpy(dtype=float)

    model_loss = log_loss(y, model_probs, labels=["H", "D", "A"])
    baseline_loss = log_loss(y, baseline_probs, labels=["H", "D", "A"])

    model_pred = np.array(["H", "D", "A"])[np.argmax(model_probs, axis=1)]
    baseline_pred = np.array(["H", "D", "A"])[np.argmax(baseline_probs, axis=1)]

    model_accuracy = accuracy_score(y, model_pred)
    baseline_accuracy = accuracy_score(y, baseline_pred)

    print("Completed matches:", len(completed))
    print(f"Model log loss: {model_loss:.4f}")
    print(f"Baseline log loss: {baseline_loss:.4f}")
    print(f"Model accuracy: {model_accuracy:.3%}")
    print(f"Baseline accuracy: {baseline_accuracy:.3%}")
    print(f"Relative log-loss improvement: {(baseline_loss - model_loss) / baseline_loss:.2%}")

    print("\nModel classification report:")
    print(classification_report(
        y, model_pred, labels=["H", "D", "A"], zero_division=0
    ))

    cm = confusion_matrix(y, model_pred, labels=["H", "D", "A"])
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Home", "Draw", "Away"]
    ).plot()
    plt.show()

    return {
        "Matches": len(completed),
        "Model log loss": model_loss,
        "Baseline log loss": baseline_loss,
        "Model accuracy": model_accuracy,
        "Baseline accuracy": baseline_accuracy
    }


# Run this after entering actual results:
# live_metrics = evaluate_live_predictions()

NameError: name 'PREDICTIONS_PATH' is not defined

## 8. Live-testing protocol

1. Before kickoff, add fixtures in section 4.
2. Run section 5 and check that predictions have been saved.
3. Do not edit the probabilities after kickoff.
4. After the matches finish, enter the final scores in section 6 and run `add_actual_results(...)`.
5. Run section 7 to evaluate all completed predictions.
6. Keep `live_predictions.csv` backed up.

**Interpretation:** this is a prospective model evaluation, not evidence of profitability. If you later want to study betting returns, that requires reliable historical/live odds, bookmaker margin adjustment, and a separate methodology.